# Math 432 — Ridge Regression (Student Notebook)

We study ridge regression on a small, approximately linear dataset. You'll implement ridge by hand using linear algebra and explore how the regularization parameter $\lambda$ changes both the fit and generalization.

**Closed-form ridge:**
$$\mathbf{w}_\lambda = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}.$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(432)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate a toy dataset

In [ ]:
n_total = 20
x = rng.uniform(-2.0, 2.0, size=n_total)
noise = rng.normal(0.0, 0.25, size=n_total)
true_w = np.array([-1.0, 2.0])  # intercept -1, slope 2

y = true_w[0] + true_w[1]*x + noise

idx = np.arange(n_total)
rng.shuffle(idx)
train_size = 12
train_idx, test_idx = idx[:train_size], idx[train_size:]
x_train, y_train = x[train_idx], y[train_idx]
x_test,  y_test  = x[test_idx],  y[test_idx]

print(f"Train size: {len(x_train)}, Test size: {len(x_test)}")

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(x_train, y_train, label='Train')
ax.scatter(x_test, y_test, label='Test')
x_line = np.linspace(-2, 2, 200)
y_line = true_w[0] + true_w[1]*x_line
ax.plot(x_line, y_line, 'k--', label='True linear')
ax.set_title('Small, approximately linear dataset (with noise)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(); plt.show()

## 2. Standardize and build polynomial features
We standardize inputs using training statistics: $x_{\text{std}} = (x - \mu)/\sigma$, then build $\phi(x) = [1, x, x^2, \dots, x^D]^T$.

In [ ]:
def poly_features(x, D):
    x = np.asarray(x)
    return np.vstack([x**d for d in range(D+1)]).T

# Standardize using training set only
x_mu = float(np.mean(x_train))
x_std = float(np.std(x_train))

def standardize(x):
    return (x - x_mu) / (x_std + 1e-12)

x_train_std = standardize(x_train)
x_test_std  = standardize(x_test)

D = 12
X_train = poly_features(x_train_std, D)
X_test  = poly_features(x_test_std,  D)
print('Shapes:', X_train.shape, X_test.shape)

## 3. Exercise 1: Implement ridge by hand
Use `np.linalg.solve`.

In [ ]:
def ridge_fit(X, y, lam, lam_min=1e-8):
    """Return w solving (X^T X + lam I) w = X^T y (with small floor for stability)."""
    y = np.asarray(y).reshape(-1)
    n, p = X.shape
    lam_eff = max(float(lam), lam_min)
    ### cutoff lam for stability: use lam_eff instead of lam
    ### TODO:
    # A = ...
    # b = ...
    try:
        w = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        w, *_ = np.linalg.lstsq(A, b, rcond=None)
    return w

def ridge_predict(X, w):
    """Return predictions X*w."""
    ### TODO:
    # return ...

In [ ]:
def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

lams = np.logspace(-6, 2, 60)

## 4) Train/test MSE vs $\lambda$
Run after you complete `ridge_fit` and `ridge_predict`.

In [ ]:
train_mse, test_mse = [], []
for lam in lams:
    w = ridge_fit(X_train, y_train, lam)
    train_mse.append(mse(y_train, ridge_predict(X_train, w)))
    test_mse.append(mse(y_test,  ridge_predict(X_test,  w)))

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(lams, train_mse, label='Train MSE')
ax.plot(lams, test_mse,  label='Test MSE')
ax.set_xscale('log')
ax.set_xlabel(rf'$\lambda$ (log scale)'); ax.set_ylabel('MSE')
ax.set_title(rf'Train vs Test MSE across ridge $\lambda$')
ax.legend(); plt.show()

## 5. Experiment: fitted curves for chosen $\lambda$ values

Edit `selected_lams` and re-run to see the effect.

In [ ]:
selected_lams = [1e-8, 1e-6, 1e-4, 1e-3, 1e-2, 1e-1, 1.0] ## omit 0.0, but still get divide-by-0 errors?
lam_min = 1e-8

# Trim x-range further to [-1, 1]
x_plot = np.linspace(-1, 1, 400)
x_plot_std = standardize(x_plot)
X_plot = poly_features(x_plot_std, D)

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(x_train, y_train, label='Train')
ax.scatter(x_test, y_test, label='Test')
ax.plot(x_line, y_line, 'k--', label='True linear')

# Clip range for y predictions (adjust to taste)
clip_lo, clip_hi = -5.0, 5.0

for lam in selected_lams:
    w = ridge_fit(X_train, y_train, max(lam, lam_min))
    if not np.all(np.isfinite(w)):
        print(f"Skipping λ={lam:g}: non-finite weights")
        continue
    y_plot = ridge_predict(X_plot, w)
    y_plot = np.clip(y_plot, clip_lo, clip_hi)
    ax.plot(x_plot, y_plot, label=rf'$\lambda$={lam:g}')

ax.set_title(r'Ridge fits for selected $\lambda$ values (clipped)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(ncol=2)
plt.show()

## 6. Reflection
1. Why does tiny $\lambda$ yield low training MSE but poor test MSE?
2. What happens as you increase $D$ at fixed small $n$? How does ridge help?

## 7. Exercise 2: Ridge via SVD
**Implementation note:** For numerical stability we use the economy SVD and compute $\mathbf{w}_\lambda$ directly from the singular values.

In [ ]:
U, s, Vt = np.linalg.svd(X_train, full_matrices=False)
V = Vt.T

def ridge_fit_svd_stable(U, s, V, y, lam, tol=1e-12):
    y = np.asarray(y).reshape(-1)
    if lam <= tol:  # exceptional case, if lam is too small
        inv_s = np.zeros_like(s)
        mask = s > tol
        inv_s[mask] = 1.0 / s[mask]
        return V @ (np.diag(inv_s) @ (U.T @ y))
    # TODO:
    # factors = ... ### diagonal entries of middle matrix
    # return ... ### Hint: one factor should be np.diag(factors)

# Verify equality with direct ridge
test_lams = [1e-8, 1e-6, 1e-4, 1e-2, 1e-1, 1.0]
for lam_test in test_lams:
    w_direct = ridge_fit(X_train, y_train, lam_test)
    w_svd_stable = ridge_fit_svd_stable(U, s, V, y_train, lam_test)
    diff = np.linalg.norm(w_direct - w_svd_stable)
    print(f"λ={lam_test:g} -> ||w_direct - w_svd|| = {diff:.3e}")

In [ ]:
lams_plot = np.logspace(-8, 2, 120)
fig, ax = plt.subplots(figsize=(6,4))
for i, sigma in enumerate(s):
    ax.plot(lams_plot, (sigma**2)/(sigma**2 + lams_plot), label=rf'$\sigma_{i+1}$={sigma:.3f}')
ax.set_xscale('log')
ax.set_xlabel(r'$\lambda$ (log scale)')
ax.set_ylabel(r'Shrinkage factor $\sigma_i^2/(\sigma_i^2+\lambda)$')
ax.set_title('Direction-wise shrinkage vs $\lambda$')
ax.legend(); plt.show()

## 8. Exercise 3: Ridge via an augmented linear system

Ridge regression solves:
$$
\min_{\mathbf{w} \in \mathbb{R}^p} \; \| y - X\mathbf{w} \|^2 + \lambda \|\mathbf{w}\|^2,
$$

with solution:

$$
\hat{\mathbf{w}}_{\text{ridge}} = (X^\top X + \lambda I_p)^{-1} X^\top y.
$$

1. Show that this ridge regression solution can be seen as the **ordinary least-squares solution** to an **augmented linear system**.  
   Specifically, find $\tilde{X}$ and $\tilde{y}$ such that solving:
   $$
   \min_{\mathbf{w}} \; \| \tilde{y} - \tilde{X}\mathbf{w} \|^2
   $$
   gives the same solution as ridge regression.

   *Hint:* Add extra rows to $X$ and $y$ that encode the penalty term $\lambda \|\mathbf{w}\|^2$.

2. Write down $\tilde{X}$ and $\tilde{y}$ explicitly in terms of $X, y, \lambda$, and $I_p$.

3. Verify algebraically that the normal equations for this augmented system match the ridge regression normal equations.

---

### Computational Experiment

Implement this idea and check that the augmented system gives the same solution as ridge regression.


In [ ]:
# Step 1: Generate synthetic data
np.random.seed(0)
n, p = 50, 5
X = np.random.randn(n, p)
beta_true = np.array([1, -2, 3, 0, 0])
y = X @ beta_true + 0.5 * np.random.randn(n)

# Step 2: Ridge regression solution
lam = 10.0
ridge_beta = np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)

print("Ridge solution:", ridge_beta)


In [ ]:

# Step 3: Construct augmented system
### TODO: fill in your solution
# tilde_X = np.vstack([X, ...])
# tilde_y = np.concatenate([y, ...])

# Solve using ordinary least squares
augmented_beta = np.linalg.lstsq(tilde_X, tilde_y, rcond=None)[0]

print("Augmented system solution:", augmented_beta)
print("Difference:", np.linalg.norm(ridge_beta - augmented_beta))



**Expected result:**  
The difference should be extremely small (close to machine precision), confirming that ridge regression can be viewed as ordinary least squares on an augmented system.

---

**Reflection:**  
Explain why this interpretation is useful for understanding ridge regression and how it connects to the idea of “data augmentation” in regularization.
